In [1]:
import os
import re
import torch
import librosa
import pandas as pd
import evaluate
from pathlib import Path
from tqdm.auto import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration

/mnt/data-disk/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
MODEL_DIR = "/mnt/data-disk/home/Ibsa-projects/ASR-Folder/ASR_experiment/Amharic_ASR_whisper/Outputs/whisper_small_amharic_normalized_v3_continue/final_model"

TEST_CSV = "/mnt/data-disk/home/Ibsa-projects/ASR-Folder/ASR_experiment/Amharic_ASR_whisper/Dataset/Training_Dataset/merged_test_dataset.csv"

AUDIO_PATH_COLUMN = "audio_path"
TEXT_COLUMN = "transcript"

TARGET_SR = 16000
LANGUAGE = "amharic"
TASK = "transcribe"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Model exists:", os.path.exists(MODEL_DIR))
print("Test CSV exists:", os.path.exists(TEST_CSV))

Device: cuda
Model exists: True
Test CSV exists: True


In [9]:
processor = WhisperProcessor.from_pretrained(
    MODEL_DIR,
    language=LANGUAGE,
    task=TASK,
)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_DIR)
model.to(DEVICE)
model.eval()

# Make generation explicit.
forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE,
    task=TASK,
)

model.generation_config.forced_decoder_ids = forced_decoder_ids
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK

# Safer for transcription without timestamps.
model.generation_config.return_timestamps = False

print("Model loaded successfully.")

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 8214.14it/s]


Model loaded successfully.


In [10]:
def normalize_amharic_for_asr(text: str) -> str:
    text = str(text)

    text = re.sub(r"\s+", " ", text).strip()

    replacements = {
        "ሐ": "ሀ", "ሑ": "ሁ", "ሒ": "ሂ", "ሓ": "ሃ", "ሔ": "ሄ", "ሕ": "ህ", "ሖ": "ሆ",
        "ኀ": "ሀ", "ኁ": "ሁ", "ኂ": "ሂ", "ኃ": "ሃ", "ኄ": "ሄ", "ኅ": "ህ", "ኆ": "ሆ",
        "ዐ": "አ", "ዑ": "ኡ", "ዒ": "ኢ", "ዓ": "አ", "ዔ": "ኤ", "ዕ": "እ", "ዖ": "ኦ",
        "ጸ": "ፀ", "ጹ": "ፁ", "ጺ": "ፂ", "ጻ": "ፃ", "ጼ": "ፄ", "ጽ": "ፅ", "ጾ": "ፆ",
    }

    for src, tgt in replacements.items():
        text = text.replace(src, tgt)

    text = re.sub(r"[።፣፤፥፦፧፨.,!?;:\"'()\[\]{}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_training_transcript(text: str) -> str:
    text = normalize_amharic_for_asr(text)

    text = re.sub(r"\bየ\s+", "የ", text)
    text = re.sub(r"\bበ\s+", "በ", text)
    text = re.sub(r"\bለ\s+", "ለ", text)
    text = re.sub(r"\bእንደ\s+", "እንደ", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text

In [11]:
def transcribe_audio(audio_path: str,*, max_new_tokens: int = 444, 
                    num_beams: int = 1,) -> str:

    audio_path = str(audio_path)

    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    speech, sr = librosa.load(audio_path, sr=TARGET_SR, mono=True)

    inputs = processor.feature_extractor(
        speech,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        return_attention_mask=True,
    )

    input_features = inputs.input_features.to(DEVICE)

    attention_mask = None
    if hasattr(inputs, "attention_mask"):
        attention_mask = inputs.attention_mask.to(DEVICE)

    with torch.no_grad():
        generated_ids = model.generate(
            input_features=input_features,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            do_sample=False,
            language=LANGUAGE,
            task=TASK,
            return_timestamps=False,
        )

    text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )[0]

    # return normalize_training_transcript(text)
    return text

In [22]:
from IPython.display import Audio, display
from random import randint

random_index = randint(0, len(pd.read_csv(TEST_CSV)) - 1)

sample_audio = pd.read_csv(TEST_CSV).iloc[random_index][AUDIO_PATH_COLUMN]
sample_ref = pd.read_csv(TEST_CSV).iloc[random_index][TEXT_COLUMN]

pred = transcribe_audio(sample_audio, num_beams=1)

print(f"Randomly selected index: {random_index}")

print("AUDIO:")
print(sample_audio)

display(Audio(sample_audio, rate=TARGET_SR))

print("\nREFERENCE:")
print(normalize_training_transcript(sample_ref))

print("\nPREDICTION:")
print(pred)

[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Randomly selected index: 3640
AUDIO:
/mnt/data-disk/home/Ibsa-projects/ASR-Folder/Amharic_speech_recognition_using_local_dataset_code_scripts/Dataset/Dataset_with_path/Dataset/test/yfwVOF6ah45UBUGEqd6p.wav



REFERENCE:
በቀሎ በቀሎ ምንም እንኳን የምግብ ሰብል ቢሆንም በከፍተኛ መጠን ለሽያጭና ለእንስሳት መኖ እንደጥሬ ሰብል ውስጥ ይመደባል

PREDICTION:
በቆሎ ወይም ይዝ ወይም ኮርም ምንም እንኳን በቆሎ ዋነኛ አህመገብ ሰብል ቢሆንም በከፍተኛ መጠን ለሽያጭ እና ደንስ ሰአት ምንውነት የሚውል እንደጥሬ ሰብር ይቆጠራል
